In [ ]:
import os
from pathlib import Path

import pandas as pd
from typing import List, Dict

In [ ]:
# __file__ = str(Path(".").absolute() / "interaction_scores.ipynb")
# __file__
NB_DIR = Path(".").absolute() / "interaction_scores.ipynb"
NB_DIR

In [ ]:
print(os.getcwd())
# _ROOT = Path(os.environ.get("HOME_PROJ_DIR", Path(__file__).resolve().parents[2]))
_ROOT = Path(os.environ.get("HOME_PROJ_DIR", NB_DIR.resolve().parents[2]))
os.chdir(_ROOT)
print(os.getcwd())

In [ ]:
interactions_dir = _ROOT / "data" / "plip_kinodata3d" / "processed"
interaction_names = ["halogen_bonds", "hydrogen_bonds", "hydrophobic_interactions", "pi_stacking", "salt_bridges"]

mapping = pd.read_csv(_ROOT / "data" / "ident_to_activity_id.csv")
mapping.rename(columns={"activities.activity_id": "activity_id"}, inplace=True)

halogen_bonds_df = pd.read_csv(interactions_dir / "halogen_bonds.csv")
hydrogen_bonds_df = pd.read_csv(interactions_dir / "hydrogen_bonds.csv")
hydrophobic_interactions_df = pd.read_csv(interactions_dir / "hydrophobic_interactions.csv")
pi_stacking_df = pd.read_csv(interactions_dir / "pi_stacking.csv")
salt_bridges_df = pd.read_csv(interactions_dir / "salt_bridges.csv")

# interactions = [pd.read_csv(interactions_dir / f) for f in interaction_files]
interactions_dfs = [halogen_bonds_df,
                hydrogen_bonds_df,
                hydrophobic_interactions_df,
                pi_stacking_df,
                salt_bridges_df]

# Intercation Score

The final score will be calculated as:
$$
\displaystyle\sum_{\text{residues}}{\text{residue attribution score} × \frac{\displaystyle\sum_{\text{bonds}}{\text{bond score}}}{\text{total number of bonds the ligand CAN form}}}
$$

### H-bonds
What PLIP does:
- Donor–acceptor distance $d_{DA}$ is $≤ 4.1 Å$ `(HBOND_DIST_MAX = 4.1)`
- Donor angle $∠(D−H⋯A)$ is $≥ 100°$ `(HBOND_DON_ANGLE_MIN = 100)`
- If the same groups already form a salt bridge, PLIP removes the overlapping hydrogen bond(s) (to avoid double-reporting)
- A donor is restricted to one H-bond; if multiple acceptors are possible, PLIP keeps the one whose $∠(D−H⋯A)$ is closest to $180°$. Acceptors can still participate in multiple bonds (bifurcated cases).

Let's define:

**Q**: a quality score that does not change too abruptly near the geometric thresholds, using a smooth logistic formulation to do this and to keep it in $[0,1]$:
$$
s_\theta = \sigma\!\left(\frac{d_{\max} - d}{\lambda_d}\right) \text{, and } s_d = \sigma\!\left(\frac{\theta - \theta_{\min}}{\lambda_\theta}\right)
$$
$$ 
Q = s_\theta \cdot s_d
$$

where the logistic (sigmoid) function is defined as:
$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

**Parameters:**

- $ d $: donor–acceptor distance (Å)  
- $ \theta $: donor–acceptor angle (degrees)  
- $ d_{\max} $: maximum distance cutoff (e.g. 4.1 Å in PLIP)  
- $ \theta_{\min} $: minimum angle cutoff (e.g. 100° in PLIP)  
- $ \lambda_d $: controls how fast the distance contribution ramps  
  (here **0.2–0.4 Å**)  
- $ \lambda_\theta $: controls angular softness  
  (here **5–15°**)  

The resulting score $Q \in (0,1)$, with higher values corresponding to shorter and more linear hydrogen bonds.